# 03 - Autoregressive Object Decoder

## Objective

The previous notebook created a dataset of frozen, L2-normalized CLIP text
embeddings paired with object nouns.

A temporary linear classifier was used only to verify that the embeddings and
labels were correctly aligned.

This notebook replaces that classifier with a NOVIC-style autoregressive object
decoder.

The model will learn:

CLIP embedding
→ prefix projection
→ Transformer decoder
→ object noun tokens
→ EOS

The main goals are:

1. Convert target object nouns into token sequences.
2. Define EOS and PAD handling.
3. Project one CLIP embedding into multiple prefix vectors.
4. Implement a decoder-only Transformer.
5. Implement causal attention.
6. Implement teacher forcing.
7. Implement shifted token prediction.
8. Overfit a tiny batch of examples.
9. Verify autoregressive noun generation.

The tiny-batch overfitting experiment is the main completion criterion for this
notebook.

## 1. Environment setup

In [85]:
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [86]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [87]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [88]:
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/mini_novic_storage"
)

EMBEDDING_CACHE = (
    DRIVE_ROOT
    / "embeddings"
    / "text_embedding_dataset"
    / "clip_text_embeddings.pt"
)

CHECKPOINT_DIR = (
    DRIVE_ROOT
    / "checkpoints"
    / "autoregressive_decoder"
)

RESULT_DIR = (
    DRIVE_ROOT
    / "results"
    / "autoregressive_decoder"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Embedding cache:", EMBEDDING_CACHE)

Embedding cache: /content/drive/MyDrive/mini_novic_storage/embeddings/text_embedding_dataset/clip_text_embeddings.pt


## 2. Load cached CLIP embeddings

The frozen CLIP text embeddings were generated in Notebook 02.

CLIP does not need to be executed again during decoder training.

The decoder therefore receives precomputed semantic vectors directly.

In [89]:
cache = torch.load(
    EMBEDDING_CACHE,
    map_location="cpu"
)

embeddings = cache["embeddings"]
noun_texts = cache["noun_text"]
splits = cache["split"]

print("Embeddings:", embeddings.shape)
print("Number of targets:", len(noun_texts))
print("CLIP model:", cache["model_name"])
print("Pretrained:", cache["pretrained"])

Embeddings: torch.Size([2500, 512])
Number of targets: 2500
CLIP model: ViT-B-32
Pretrained: laion2b_s34b_b79k


Check normalization:

In [90]:
norms = embeddings.norm(dim=-1)

print("Minimum norm:", norms.min().item())
print("Maximum norm:", norms.max().item())
print("Mean norm:", norms.mean().item())

Minimum norm: 0.9999997615814209
Maximum norm: 1.000000238418579
Mean norm: 1.0


## 3. Target noun tokenizer

For the first decoder implementation we use a small word-level vocabulary built
from the object nouns.

Special tokens:

- `<PAD>` — padding
- `<EOS>` — end of generated noun

In [91]:
unique_nouns = sorted(
    set(noun_texts)
)

print("Number of nouns:", len(unique_nouns))
print(unique_nouns[:20])

Number of nouns: 50
['airplane', 'backpack', 'baseball bat', 'bear', 'bed', 'bicycle', 'bird', 'boat', 'book', 'bottle', 'bus', 'calculator', 'camera', 'car', 'cat', 'chair', 'clock', 'coffee mug', 'cow', 'cup']


Build word vocabulary:

In [92]:
special_tokens = [
    "<PAD>",
    "<EOS>"
]

words = set()

for noun in unique_nouns:
    for word in noun.split():
        words.add(word)

vocab = (
    special_tokens
    + sorted(words)
)

token_to_id = {
    token: idx
    for idx, token in enumerate(vocab)
}

id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}

PAD_ID = token_to_id["<PAD>"]
EOS_ID = token_to_id["<EOS>"]

VOCAB_SIZE = len(vocab)

print("Vocabulary size:", VOCAB_SIZE)
print("PAD ID:", PAD_ID)
print("EOS ID:", EOS_ID)

Vocabulary size: 58
PAD ID: 0
EOS ID: 1


### Tokenization example
Create:

In [93]:
def encode_noun(noun):
    tokens = [
        token_to_id[word]
        for word in noun.split()
    ]

    tokens.append(EOS_ID)

    return tokens

Test:

In [94]:
for noun in [
    "dog",
    "fire truck",
    "sports car",
    "coffee mug"
]:
    print(
        noun,
        "→",
        encode_noun(noun)
    )

dog → [24, 1]
fire truck → [26, 54, 1]
sports car → [47, 16, 1]
coffee mug → [20, 38, 1]


### Decode tokens
We'll need the reverse operation for inference.

In [95]:
def decode_tokens(token_ids):

    words = []

    for token_id in token_ids:

        if token_id == EOS_ID:
            break

        if token_id == PAD_ID:
            continue

        words.append(
            id_to_token[token_id]
        )

    return " ".join(words)

Test:

In [96]:
encoded = encode_noun(
    "fire truck"
)

print("Encoded:", encoded)
print("Decoded:", decode_tokens(encoded))

Encoded: [26, 54, 1]
Decoded: fire truck


## Determine maximum target length

In [97]:
encoded_targets = [
    encode_noun(noun)
    for noun in noun_texts
]

MAX_TARGET_LEN = max(
    len(sequence)
    for sequence in encoded_targets
)

print(
    "Maximum target length:",
    MAX_TARGET_LEN
)


Maximum target length: 3


In [98]:
MAX_SEQ_LEN = max(
    MAX_TARGET_LEN,
    4
)

print("Sequence length:", MAX_SEQ_LEN)

Sequence length: 4


## Padding

In [99]:
def pad_sequence(sequence):

    sequence = sequence[
        :MAX_SEQ_LEN
    ]

    padding_needed = (
        MAX_SEQ_LEN
        - len(sequence)
    )

    sequence = (
        sequence
        + [PAD_ID] * padding_needed
    )

    return sequence

Test:

In [100]:
for noun in [
    "dog",
    "fire truck"
]:

    encoded = encode_noun(noun)
    padded = pad_sequence(encoded)

    print(noun)
    print("encoded:", encoded)
    print("padded :", padded)
    print()

dog
encoded: [24, 1]
padded : [24, 1, 0, 0]

fire truck
encoded: [26, 54, 1]
padded : [26, 54, 1, 0]



## Build all target sequences

In [101]:
target_sequences = torch.tensor(
    [
        pad_sequence(
            encode_noun(noun)
        )
        for noun in noun_texts
    ],
    dtype=torch.long
)

print(
    target_sequences.shape
)

torch.Size([2500, 4])


Inspect:

In [102]:
for i in range(10):

    print(
        noun_texts[i],
        "→",
        target_sequences[i].tolist()
    )

dog → [24, 1, 0, 0]
dog → [24, 1, 0, 0]
dog → [24, 1, 0, 0]
dog → [24, 1, 0, 0]
dog → [24, 1, 0, 0]
dog → [24, 1, 0, 0]
dog → [24, 1, 0, 0]
dog → [24, 1, 0, 0]
dog → [24, 1, 0, 0]
dog → [24, 1, 0, 0]


## Decoder configuration

In [103]:
CLIP_DIM = embeddings.shape[1]

PREFIX_LEN = 4
HIDDEN_DIM = 256

NUM_LAYERS = 4
NUM_HEADS = 4

DROPOUT = 0.1

### Prefix Projection

Each CLIP embedding has **512 dimensions**:

\[
z \in \mathbb{R}^{512}.
\]

This single 512-dimensional vector contains the semantic information produced by CLIP for the corresponding noun.

For example:

```text
"fire truck"
      ↓
CLIP text encoder
      ↓
512-dimensional CLIP embedding
```

In Notebook 03, the CLIP embedding is used as the **conditioning information** for the autoregressive Transformer decoder.

The Transformer does not directly use the 512-dimensional CLIP vector. Instead, we first convert it into several **prefix vectors** that have the same dimension as the Transformer's internal hidden representation.

---

#### Transformer Hidden Dimension

For our initial Mini-NOVIC decoder, we choose

\[
d_{\text{model}} = 256.
\]

The value **256 is a design choice (hyperparameter)**. It is not required by CLIP or by the Transformer architecture.

We could choose other values such as:

```text
128
256
512
768
```

A smaller hidden dimension usually means:

```text
fewer parameters
→ lower memory usage
→ faster training
→ lower model capacity
```

while a larger hidden dimension usually means:

```text
more parameters
→ higher memory usage
→ slower training
→ greater model capacity
```

For our first tiny-batch overfitting experiment, **256 is a reasonable small starting value**.

Another useful property is that we use **4 attention heads**, and

\[
\frac{256}{4} = 64,
\]

so each attention head can operate on 64 dimensions.

---

#### Why must each prefix vector have 256 dimensions?

The Transformer processes a sequence of vectors, and **every vector in that sequence must have the same dimension**.

Later, the Transformer input will look conceptually like:

```text
[prefix 1] [prefix 2] [prefix 3] [prefix 4] [token 1] [token 2] ...
```

Since the Transformer hidden size is 256, every element must have 256 values:

```text
Prefix 1         → 256 dimensions
Prefix 2         → 256 dimensions
Prefix 3         → 256 dimensions
Prefix 4         → 256 dimensions

Token embedding  → 256 dimensions
Token embedding  → 256 dimensions
```

Therefore,

\[
\text{prefix dimension}
=
\text{token embedding dimension}
=
d_{\text{model}}
=
256.
\]

The value 256 is therefore not specifically required for a prefix. It is used because **our Transformer was designed with a hidden size of 256**.

If we instead created a Transformer with

\[
d_{\text{model}}=512,
\]

then each prefix vector and each token embedding would also need to have 512 dimensions.

---

#### Why use 4 prefix vectors?

Instead of converting the CLIP embedding into only one Transformer vector, we convert it into **4 prefix vectors**:

```text
CLIP embedding
      ↓
[p1] [p2] [p3] [p4]
```

The number **4 is also a hyperparameter**. There is no mathematical rule saying that the prefix length must be 4.

We could choose, for example:

```text
Prefix length = 1
Prefix length = 2
Prefix length = 4
Prefix length = 8
```

Using several prefix positions gives the Transformer multiple positions through which it can receive information derived from the CLIP embedding.

For this initial model, a prefix length of **4** provides multiple conditioning positions while keeping the decoder relatively small.

---

#### Projection Operation

We therefore start with one CLIP embedding:

```text
CLIP embedding
[512]
```

and apply a learned linear projection:

```text
512 → 4 × 256
```

Since

\[
4\times256=1024,
\]

the linear layer produces 1024 values:

```text
CLIP embedding
[512]

      ↓

Linear projection

512 → 1024

      ↓

Reshape

1024 → [4, 256]
```

Therefore,

\[
z \in \mathbb{R}^{512}
\]

is converted into

\[
P \in \mathbb{R}^{4\times256}.
\]

The resulting four vectors are:

\[
P =
[p_1,p_2,p_3,p_4],
\]

where

\[
p_i \in \mathbb{R}^{256}.
\]

For a batch of samples, the shape becomes:

```text
[batch_size, 4, 256]
```

---

#### Important: Where do the prefix vectors come from?

The prefix vectors are **not taken from the noun tokens**.

They are generated entirely from the corresponding CLIP embedding:

```text
CLIP embedding
      ↓
Linear projection
      ↓
p1, p2, p3, p4
```

The noun text is used separately as the **target output**.

For example, for:

```text
"fire truck"
```

we have two sides:

```text
CONDITION / INPUT SIDE

CLIP embedding
      ↓
Linear projection
      ↓
[p1] [p2] [p3] [p4]
```

and:

```text
TARGET SIDE

"fire truck"
      ↓
["fire", "truck", "<EOS>"]
      ↓
[token IDs]
```

During training, these two parts are brought together:

```text
CLIP embedding
      ↓
[p1] [p2] [p3] [p4]
                  +
          noun token embeddings
                  ↓
             Transformer
                  ↓
       predict next noun token
```

For example:

```text
CLIP embedding for "fire truck"
             ↓
      p1  p2  p3  p4
             ↓
        Transformer
             ↓
           "fire"
             ↓
           "truck"
             ↓
           <EOS>
```

The prefix therefore acts as the **conditioning information** that tells the autoregressive decoder what semantic concept it should convert into language.

---

### Initial Mini-NOVIC Configuration

For the first decoder experiment, we use:

```text
CLIP embedding size       = 512
Prefix length             = 4
Prefix vector dimension   = 256
Transformer hidden size   = 256
Transformer layers        = 4
Attention heads           = 4
```

The important relationship is:

```text
512-dimensional CLIP embedding
              ↓
       Linear projection
              ↓
        4 × 256 values
              ↓
          reshape
              ↓
[p1] [p2] [p3] [p4]
 256  256  256  256
              ↓
      Transformer
      d_model = 256
```

Both **4** and **256** are initial architectural hyperparameters. They are not fixed mathematical requirements. Once the decoder works correctly, different prefix lengths and hidden dimensions can be tested experimentally.

## 4. CLIP embedding → prefix vectors

A single CLIP embedding is projected into several Transformer conditioning
vectors.

For prefix length P and Transformer hidden dimension H:

CLIP embedding
→ Linear(F, P × H)
→ reshape
→ [batch, P, H]

In [104]:
class PrefixMapper(nn.Module):

    def __init__(
        self,
        clip_dim,
        hidden_dim,
        prefix_len
    ):

        super().__init__()

        self.hidden_dim = hidden_dim
        self.prefix_len = prefix_len

        self.projection = nn.Linear(
            clip_dim,
            prefix_len * hidden_dim
        )

    def forward(self, x):

        batch_size = x.size(0)

        x = self.projection(x)

        x = x.view(
            batch_size,
            self.prefix_len,
            self.hidden_dim
        )

        return x

Test: ( 2 samples )

In [105]:
prefix_mapper = PrefixMapper(
    CLIP_DIM,
    HIDDEN_DIM,
    PREFIX_LEN
)

sample_embedding = embeddings[:2]

prefix = prefix_mapper(
    sample_embedding
)

print("Input :", sample_embedding.shape)
print("Output:", prefix.shape)

Input : torch.Size([2, 512])
Output: torch.Size([2, 4, 256])


## Token embeddings

In [106]:
token_embedding = nn.Embedding(
    VOCAB_SIZE,
    HIDDEN_DIM
)

test : ( 2 samples)

In [107]:
sample_targets = target_sequences[:2]

embedded_tokens = token_embedding(
    sample_targets
)

print(
    sample_targets.shape
)

print(
    embedded_tokens.shape
)

torch.Size([2, 4])
torch.Size([2, 4, 256])


## Positional embeddings

In [108]:
TOTAL_LEN = (
    PREFIX_LEN
    + MAX_SEQ_LEN
)

In [109]:
position_embedding = nn.Embedding(
    TOTAL_LEN,
    HIDDEN_DIM
)

## 9. Mini-NOVIC Autoregressive Decoder

The decoder receives two main inputs:

1. A **frozen CLIP embedding**, which tells the model what semantic concept should be generated.
2. Previous **noun tokens**, which provide the autoregressive language context.

The complete flow is:

```text
Frozen CLIP embedding
     [batch, 512]
            ↓
     Prefix Mapper
            ↓
     [batch, 4, 256]

Token IDs
   [batch, T]
            ↓
    Token Embedding
            ↓
   [batch, T, 256]

            ↓
       concatenate

[P1][P2][P3][P4][T1][T2][T3]...
            ↓
   Add positional embeddings
            ↓
   Apply causal attention mask
            ↓
     Transformer Layer 1
            ↓
     Transformer Layer 2
            ↓
     Transformer Layer 3
            ↓
     Transformer Layer 4
            ↓
   Keep token positions only
            ↓
     Output Projection
            ↓
[batch, T, vocabulary_size]
            ↓
       next-token logits
```

### 1. Prefix Mapper

```python
self.prefix_mapper = nn.Linear(
    clip_dim,
    prefix_len * hidden_dim
)
```

The frozen 512-dimensional CLIP embedding is converted into four 256-dimensional prefix vectors:

```text
[batch, 512]
      ↓
Linear
      ↓
[batch, 1024]
      ↓
reshape
      ↓
[batch, 4, 256]
```

The four prefix vectors provide the CLIP semantic information to the Transformer.

---

### 2. Token Embedding

```python
self.token_embedding = nn.Embedding(
    vocab_size,
    hidden_dim
)
```

Noun tokens are represented by integer IDs.

For example:

```text
7  → fire
14 → truck
1  → <EOS>
```

`nn.Embedding` converts every token ID into a learned 256-dimensional vector:

```text
token ID
   ↓
learned embedding
   ↓
256-dimensional vector
```

---

### 3. Positional Embedding

```python
self.position_embedding = nn.Embedding(
    prefix_len + max_seq_len,
    hidden_dim
)
```

The Transformer needs information about the order of the sequence.

For example:

```text
Position:   0   1   2   3   4   5   6
            ↓   ↓   ↓   ↓   ↓   ↓   ↓
Content:   P1  P2  P3  P4  T1  T2  T3
```

A learned position vector is added to each content vector:

\[
x
=
\text{content embedding}
+
\text{position embedding}.
\]

Therefore, the model knows both **what the vector represents** and **where it occurs in the sequence**.

---

### 4. Causal Attention Mask

The Transformer receives a sequence such as:

```text
P1 P2 P3 P4 T1 T2 T3
```

The prefix vectors can freely attend to each other:

```text
P1 ↔ P2 ↔ P3 ↔ P4
```

A noun token can see:

```text
all prefix vectors
+
previous noun tokens
+
its current position
```

but it cannot see future noun tokens.

Conceptually:

```text
         P1 P2 P3 P4 T1 T2 T3

P1       ✓  ✓  ✓  ✓  ✗  ✗  ✗
P2       ✓  ✓  ✓  ✓  ✗  ✗  ✗
P3       ✓  ✓  ✓  ✓  ✗  ✗  ✗
P4       ✓  ✓  ✓  ✓  ✗  ✗  ✗

T1       ✓  ✓  ✓  ✓  ✓  ✗  ✗
T2       ✓  ✓  ✓  ✓  ✓  ✓  ✗
T3       ✓  ✓  ✓  ✓  ✓  ✓  ✓
```

In the numerical mask:

```text
0     = attention allowed
-inf  = attention blocked
```

This prevents the model from looking at future correct tokens during training.

Without this mask, the model could cheat.

---

### 5. Transformer

```python
self.transformer = nn.TransformerEncoder(...)
```

Although PyTorch calls this a `TransformerEncoder`, it is a stack of self-attention Transformer blocks.

By supplying our own causal attention mask, we use these blocks as a **decoder-only autoregressive Transformer**.

Each Transformer layer contains approximately:

```text
Input
  ↓
Layer Normalization
  ↓
Multi-Head Self-Attention
  ↓
Residual Connection
  ↓
Layer Normalization
  ↓
Feed-Forward Network
  ↓
Residual Connection
  ↓
Output
```

Self-attention allows the noun tokens to use information from the CLIP prefixes and previous noun tokens.

The model uses four Transformer layers, so the representation is refined several times before prediction.

---

### 6. Output Projection

The Transformer still produces 256-dimensional vectors.

We need to convert each token-position vector into scores for the vocabulary:

```python
self.output_projection = nn.Linear(
    hidden_dim,
    vocab_size
)
```

For example, with a vocabulary size of 58:

```text
256-dimensional vector
        ↓
Linear
        ↓
58 scores
```

These scores are called **logits**.

For example:

```text
<PAD>  → -2.4
<EOS>  →  0.7
cat    → -1.1
fire   →  5.9
truck  →  1.3
...
```

The logits are later compared with the correct next token when calculating the training loss.

---

### Why Are Prefix Outputs Removed Before Prediction?

The Transformer processes the complete sequence:

```text
[P1][P2][P3][P4][T1][T2][T3]
```

but we only want noun-token predictions.

Therefore:

```python
token_outputs = x[:, self.prefix_len:, :]
```

removes the four prefix positions from the final prediction stage.

The remaining tensor contains only:

```text
[T1][T2][T3]...
```

These vectors are sent to the vocabulary output layer.

---

### Trainable Parts

The following parameters are learned during decoder training:

```text
Prefix projection          ✓ trainable
Token embeddings           ✓ trainable
Positional embeddings      ✓ trainable
Self-attention layers      ✓ trainable
Feed-forward layers        ✓ trainable
Output projection          ✓ trainable

CLIP encoder               ✗ frozen
Stored CLIP embeddings     ✗ fixed
```

Because all layers are stored inside `MiniNOVICDecoder` using `self.<layer_name>`, PyTorch automatically includes their parameters in:

```python
model.parameters()
```

Therefore, the optimizer can train the complete decoder together.

The overall goal of the model is:

```text
CLIP semantic embedding
          ↓
    prefix vectors
          ↓
autoregressive Transformer
          ↓
     noun tokens
          ↓
        <EOS>
```

In [110]:
class MiniNOVICDecoder(nn.Module):

    def __init__(
        self,
        clip_dim,
        vocab_size,
        hidden_dim=256,
        prefix_len=4,
        max_seq_len=4,
        num_layers=4,
        num_heads=4,
        dropout=0.1,
    ):
        super().__init__()

        # --------------------------------------------------
        # Save important model dimensions
        # --------------------------------------------------
        self.hidden_dim = hidden_dim
        self.prefix_len = prefix_len
        self.max_seq_len = max_seq_len
        self.vocab_size = vocab_size

        # --------------------------------------------------
        # 1. CLIP embedding -> prefix vectors
        #
        # [batch, 512]
        #       ->
        # [batch, 4 * 256]
        #       ->
        # [batch, 4, 256]
        # --------------------------------------------------
        self.prefix_mapper = nn.Linear(
            clip_dim,
            prefix_len * hidden_dim
        )

        # --------------------------------------------------
        # 2. Token IDs -> token vectors
        #
        # token ID -> 256-dimensional vector
        # --------------------------------------------------
        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_dim
        )

        # --------------------------------------------------
        # 3. Positional embeddings
        #
        # Positions are required for:
        # [prefixes + noun tokens]
        # --------------------------------------------------
        self.position_embedding = nn.Embedding(
            prefix_len + max_seq_len,
            hidden_dim
        )

        # --------------------------------------------------
        # 4. One Transformer layer
        # --------------------------------------------------
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        # --------------------------------------------------
        # 5. Stack several Transformer layers
        # --------------------------------------------------
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # --------------------------------------------------
        # 6. Transformer output -> vocabulary logits
        #
        # 256 -> vocab_size
        # --------------------------------------------------
        self.output_projection = nn.Linear(
            hidden_dim,
            vocab_size
        )

    # ======================================================
    # Attention mask
    # ======================================================

    def build_attention_mask(
        self,
        token_len,
        device
    ):

        total_len = self.prefix_len + token_len

        # Initially block every connection
        mask = torch.full(
            (total_len, total_len),
            float("-inf"),
            device=device
        )

        # --------------------------------------------------
        # Prefix positions can see all prefix positions
        # --------------------------------------------------
        mask[
            :self.prefix_len,
            :self.prefix_len
        ] = 0

        # --------------------------------------------------
        # Token positions can see:
        #
        # - all prefix positions
        # - previous token positions
        # - their own current position
        #
        # but NOT future token positions
        # --------------------------------------------------
        for i in range(token_len):

            row = self.prefix_len + i

            mask[
                row,
                :self.prefix_len + i + 1
            ] = 0

        return mask

    # ======================================================
    # Forward pass
    # ======================================================

    def forward(
        self,
        clip_embeddings,
        token_ids
    ):

        batch_size = clip_embeddings.size(0)
        token_len = token_ids.size(1)

        # --------------------------------------------------
        # 1. CLIP embedding -> prefix vectors
        # --------------------------------------------------

        prefix = self.prefix_mapper(
            clip_embeddings
        )

        prefix = prefix.view(
            batch_size,
            self.prefix_len,
            self.hidden_dim
        )

        # prefix shape:
        # [batch, prefix_len, hidden_dim]
        # Example:
        # [batch, 4, 256]

        # --------------------------------------------------
        # 2. Token IDs -> token vectors
        # --------------------------------------------------

        token_vectors = self.token_embedding(
            token_ids
        )

        # token_vectors shape:
        # [batch, token_len, hidden_dim]
        # Example:
        # [batch, 4, 256]

        # --------------------------------------------------
        # 3. Put prefix and token vectors together
        # --------------------------------------------------

        x = torch.cat(
            [
                prefix,
                token_vectors
            ],
            dim=1
        )

        # Example:
        #
        # [P1 P2 P3 P4 T1 T2 T3 T4]
        #
        # shape:
        # [batch, prefix_len + token_len, hidden_dim]

        # --------------------------------------------------
        # 4. Add positional information
        # --------------------------------------------------

        total_len = x.size(1)

        positions = torch.arange(
            total_len,
            device=x.device
        )

        position_vectors = self.position_embedding(
            positions
        )

        x = (
            x
            + position_vectors.unsqueeze(0)
        )

        # --------------------------------------------------
        # 5. Build causal attention mask
        # --------------------------------------------------

        attention_mask = self.build_attention_mask(
            token_len=token_len,
            device=x.device
        )

        # --------------------------------------------------
        # 6. Pass through Transformer
        # --------------------------------------------------

        x = self.transformer(
            x,
            mask=attention_mask
        )

        # x shape:
        # [batch, prefix_len + token_len, hidden_dim]

        # --------------------------------------------------
        # 7. Select shifted prediction positions
        #
        # Last prefix predicts first noun token.
        # Then each noun-token state predicts the next token.
        #
        # Transformer sequence:
        # P1 P2 P3 P4 T1 T2 T3
        #
        # Prediction states:
        #             P4 T1 T2 T3
        # --------------------------------------------------

        prediction_states = x[
            :,
            self.prefix_len - 1:,
            :
        ]

        # --------------------------------------------------
        # 8. Convert prediction states to vocabulary logits
        # --------------------------------------------------

        logits = self.output_projection(
            prediction_states
        )

        return logits

### Attention Mask

The decoder is autoregressive, so a token must not see future noun tokens during training.

The Transformer sequence has the form:

```text
[P1] [P2] [P3] [P4] [T1] [T2] [T3] ...
```

The prefix vectors contain the CLIP conditioning information.

The attention rules are:

```text
Prefix vectors:
can attend to all prefix vectors

Noun tokens:
can attend to all prefix vectors
+
previous noun tokens
+
their own current position

Future noun tokens:
blocked
```

Conceptually:

```text
         P1 P2 P3 P4 T1 T2 T3

P1       ✓  ✓  ✓  ✓  ✗  ✗  ✗
P2       ✓  ✓  ✓  ✓  ✗  ✗  ✗
P3       ✓  ✓  ✓  ✓  ✗  ✗  ✗
P4       ✓  ✓  ✓  ✓  ✗  ✗  ✗

T1       ✓  ✓  ✓  ✓  ✓  ✗  ✗
T2       ✓  ✓  ✓  ✓  ✓  ✓  ✗
T3       ✓  ✓  ✓  ✓  ✓  ✓  ✓
```

The mask uses:

```text
0     → attention allowed
-inf  → attention blocked
```

The function first creates a matrix where everything is blocked:

```python
mask = torch.full(
    (total_len, total_len),
    float("-inf")
)
```

Then it opens the allowed connections.

The prefix block is opened using:

```python
mask[:prefix_len, :prefix_len] = 0
```

Then each noun token is allowed to see all prefixes and all noun-token positions up to itself.

For example:

```text
T1 → P1 P2 P3 P4 T1

T2 → P1 P2 P3 P4 T1 T2

T3 → P1 P2 P3 P4 T1 T2 T3
```

This causal masking is important because otherwise the model could look at future correct tokens during training and cheat.

---

### Forward Pass

The `forward()` method describes how one batch moves through the decoder.

It receives:

```text
clip_embeddings
token_ids
```

For example:

```text
clip_embeddings shape = [batch, 512]

token_ids shape       = [batch, token_len]
```

The processing happens in several steps.

#### 1. CLIP embedding → prefix vectors

The frozen CLIP embedding is projected:

```text
[batch, 512]
      ↓
Linear
      ↓
[batch, 4 × 256]
      ↓
reshape
      ↓
[batch, 4, 256]
```

The result is:

```text
[P1] [P2] [P3] [P4]
```

These vectors provide the semantic conditioning information.

---

#### 2. Token IDs → token embeddings

The target token IDs are converted into learned 256-dimensional vectors:

```text
[batch, token_len]
        ↓
nn.Embedding
        ↓
[batch, token_len, 256]
```

For example:

```text
7  → "fire"  → 256-D vector
14 → "truck" → 256-D vector
```

---

#### 3. Concatenate prefix and token vectors

The prefix and token vectors are joined along the sequence dimension:

```text
[P1] [P2] [P3] [P4] [T1] [T2] [T3]
```

So:

```text
prefix:
[batch, 4, 256]

tokens:
[batch, T, 256]

combined:
[batch, 4 + T, 256]
```

---

#### 4. Add positional embeddings

Each sequence position gets a learned position vector.

```text
Position:   0   1   2   3   4   5   6
            ↓   ↓   ↓   ↓   ↓   ↓   ↓
Content:   P1  P2  P3  P4  T1  T2  T3
```

The final input is:

\[
x =
\text{content vector}
+
\text{position vector}
\]

This allows the Transformer to know both what each vector represents and where it occurs in the sequence.

---

#### 5. Build the attention mask

The causal attention mask is created using the current token length.

```text
prefixes → can see prefixes

tokens → can see prefixes + previous/current tokens

future tokens → blocked
```

---

#### 6. Transformer processing

The complete sequence and mask are passed into the Transformer:

```text
prefix + token sequence
        ↓
causal attention
        ↓
Transformer layer 1
        ↓
Transformer layer 2
        ↓
Transformer layer 3
        ↓
Transformer layer 4
```

The output keeps the same basic shape:

```text
[batch, prefix_len + token_len, 256]
```

but the vectors now contain contextual information learned from the prefixes and previous tokens.

---

#### 7. Keep only token positions

The Transformer also produces outputs for the prefix positions, but we do not need to predict vocabulary tokens from them.

Therefore:

```python
token_outputs = x[:, self.prefix_len:, :]
```

keeps only:

```text
[T1] [T2] [T3] ...
```

with shape:

```text
[batch, token_len, 256]
```

---

#### 8. Vocabulary prediction

Finally, each 256-dimensional token representation is passed through:

```python
self.output_projection
```

which performs:

```text
256
 ↓
vocab_size
```

For example, if:

```text
VOCAB_SIZE = 58
```

the final output shape is:

```text
[batch, token_len, 58]
```

Each token position now has 58 logits, one score for every possible vocabulary token.

These logits are later compared with the correct next-token targets to calculate the training loss.

---

### Complete Forward Flow

```text
CLIP embeddings
[batch, 512]
      ↓
Prefix Mapper
      ↓
[batch, 4, 256]

Token IDs
[batch, T]
      ↓
Token Embedding
      ↓
[batch, T, 256]

      ↓ concatenate

[P1][P2][P3][P4][T1][T2]...

      ↓
Add positional embeddings
      ↓
Build causal attention mask
      ↓
Transformer
      ↓
Keep token positions only
      ↓
Output projection
      ↓
[batch, T, vocab_size]
      ↓
next-token logits
```

The attention mask controls **what each position is allowed to see**, while the `forward()` method controls **how the data moves through the whole decoder**.

## Instantiate it

In [111]:
decoder = MiniNOVICDecoder(
    clip_dim=CLIP_DIM,
    vocab_size=VOCAB_SIZE,
    hidden_dim=HIDDEN_DIM,
    prefix_len=PREFIX_LEN,
    max_seq_len=MAX_SEQ_LEN,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
).to(device)

/tmp/ipykernel_6186/2757436475.py:75: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Count parameters:

In [112]:
num_parameters = sum(
    p.numel()
    for p in decoder.parameters()
)

print(
    f"Decoder parameters: "
    f"{num_parameters:,}"
)

Decoder parameters: 3,716,154


### Shifted Prediction and Target Alignment

Autoregressive training requires the decoder to predict the **next token**, not simply reproduce the token already present at the current position.

Suppose the target noun is:

```text
fire truck <EOS> <PAD>
```

The correct prediction behavior is:

```text
last prefix   → fire
fire          → truck
truck         → <EOS>
```

We do **not** want:

```text
fire  → fire
truck → truck
```

because in that case the model could simply copy the token that it already sees.

---

### Why the Last Prefix Predicts the First Token

Before the first noun token is generated, no noun token exists yet.

The decoder only has the CLIP-derived prefix vectors:

```text
[P1] [P2] [P3] [P4]
```

Therefore, the hidden state at the **last prefix position** is used to predict the first noun token:

```text
[P1] [P2] [P3] [P4]
                  ↓
                fire
```

After `fire` is available, its hidden state predicts the next token:

```text
[P1] [P2] [P3] [P4] [fire]
                          ↓
                        truck
```

Then:

```text
[P1] [P2] [P3] [P4] [fire] [truck]
                                  ↓
                                <EOS>
```

---

### Training Alignment

For the target:

```text
fire truck <EOS> <PAD>
```

the alignment should be:

```text
Hidden state        Target token

last prefix     →   fire
fire position   →   truck
truck position  →   <EOS>
EOS position    →   <PAD>   (ignored)
```

The `<PAD>` target is ignored when calculating the loss.

So the decoder learns:

```text
CLIP prefix information
        ↓
predict first noun token
        ↓
previous noun token
        ↓
predict next noun token
        ↓
continue until <EOS>
```

This is the shifted next-token prediction used by the autoregressive decoder.

The important rule is:

> The model output at a position is compared with the token that should come **after** that position, not with the token already present there.

## Tiny training set

Before using all 2000 training examples, select only 32.


In [113]:
train_indices = [
    i
    for i, split in enumerate(splits)
    if split == "train"
]

tiny_indices = train_indices[:32]

tiny_X = embeddings[
    tiny_indices
]

tiny_y = target_sequences[
    tiny_indices
]

tiny_nouns = [
    noun_texts[i]
    for i in tiny_indices
]

print("Tiny embeddings:", tiny_X.shape)
print("Tiny targets:", tiny_y.shape)

for i in range(10):
    print(
        tiny_nouns[i],
        tiny_y[i].tolist()
    )

Tiny embeddings: torch.Size([32, 512])
Tiny targets: torch.Size([32, 4])
dog [24, 1, 0, 0]
dog [24, 1, 0, 0]
dog [24, 1, 0, 0]
dog [24, 1, 0, 0]
dog [24, 1, 0, 0]
dog [24, 1, 0, 0]
dog [24, 1, 0, 0]
dog [24, 1, 0, 0]
dog [24, 1, 0, 0]
dog [24, 1, 0, 0]


### One important improvement
Our first 32 dataset rows may all correspond to the same noun because Notebook 02 generated captions noun-by-noun.

That's bad for this debugging test.

Instead choose examples across several nouns:

In [114]:
tiny_indices = []

seen_nouns = set()

for i, (
    noun,
    split
) in enumerate(
    zip(noun_texts, splits)
):

    if split != "train":
        continue

    if noun not in seen_nouns:

        tiny_indices.append(i)
        seen_nouns.add(noun)

    if len(tiny_indices) == 32:
        break

In [115]:
tiny_X = embeddings[tiny_indices]

tiny_y = target_sequences[tiny_indices]

tiny_nouns = [
    noun_texts[i]
    for i in tiny_indices
]

print(
    "Unique tiny nouns:",
    len(set(tiny_nouns))
)

Unique tiny nouns: 32


## 5. Shifted autoregressive prediction

The decoder must predict the target noun one token at a time.

For a target such as:

```text
fire truck EOS
```
the desired prediction alignment is:

```text
last prefix state   → fire  
"fire" state        → truck  
"truck" state       → EOS
```

The model must NOT see the token that it is currently trying to predict.

During training, the previous correct target tokens are supplied to the decoder.
This is called teacher forcing.

## Create decoder inputs
Suppose one padded target is:

```text
fire truck EOS PAD
```
Our complete target tensor is:

```text
[fire, truck, EOS, PAD]
```
But the token inputs given to the Transformer should only be:

```text
[fire, truck, EOS]
```
because the first target fire is predicted from the prefix.

In [116]:
decoder_inputs = tiny_y[:, :-1]

print("Full targets shape :", tiny_y.shape)
print("Decoder input shape:", decoder_inputs.shape)

Full targets shape : torch.Size([32, 4])
Decoder input shape: torch.Size([32, 3])


Inspect one:

In [117]:
example_index = 0

print("Noun:")
print(tiny_nouns[example_index])

print("\nFull target:")
print(tiny_y[example_index].tolist())

print("\nDecoder input:")
print(decoder_inputs[example_index].tolist())

Noun:
dog

Full target:
[24, 1, 0, 0]

Decoder input:
[24, 1, 0]


## Run one forward pass
Move data to GPU:

In [118]:
tiny_X_device = tiny_X.to(device)
tiny_y_device = tiny_y.to(device)

decoder_inputs = tiny_y_device[:, :-1]

In [119]:
decoder.eval()

with torch.no_grad():
    all_logits = decoder(
        tiny_X_device,
        decoder_inputs
    )

print("All logits shape:")
print(all_logits.shape)
print("Targets shape:", tiny_y_device.shape)

All logits shape:
torch.Size([32, 4, 58])
Targets shape: torch.Size([32, 4])


### Prediction alignment

For a target:
```text
fire truck <EOS> <PAD>

the model receives:

prefixes + [fire, truck, EOS]
```
and predictions are aligned as:

| Transformer state | Training target |
|---|---|
| Last prefix state | fire |
| fire state | truck |
| truck state | EOS |
| EOS state | PAD |

The PAD target is ignored when calculating the loss.

## Cross-entropy loss with PAD ignored

In [120]:
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

CrossEntropyLoss expects:
```text
logits:
[N, classes]

target:
[N]
```
so flatten the batch and sequence dimensions:

In [121]:
prediction_logits = all_logits

loss = criterion(
    prediction_logits.reshape(
        -1,
        VOCAB_SIZE
    ),
    tiny_y_device.reshape(-1)
)

print("Initial loss:", loss.item())

Initial loss: 4.652957916259766


At this point the loss can be large. That's expected — the model hasn't learned anything yet.

## Make a reusable loss function

In [122]:
def compute_decoder_loss(
    model,
    clip_embeddings,
    target_tokens,
    criterion
):

    # --------------------------------
    # Teacher-forcing input
    # --------------------------------

    token_inputs = target_tokens[:, :-1]

    # --------------------------------
    # Transformer forward pass
    # --------------------------------

    all_logits = model(
        clip_embeddings,
        token_inputs
    )

    prefix_len = model.prefix_len

    # --------------------------------
    # Last prefix predicts first token
    # --------------------------------

    first_logits = all_logits[
        :,
        prefix_len - 1 : prefix_len,
        :
    ]

    # --------------------------------
    # Each provided token predicts
    # the following target token
    # --------------------------------

    next_logits = all_logits[
        :,
        prefix_len :
        prefix_len + token_inputs.size(1),
        :
    ]

    # --------------------------------
    # [B, target_len, vocab_size]
    # --------------------------------

    logits = torch.cat(
        [
            first_logits,
            next_logits
        ],
        dim=1
    )

    # --------------------------------
    # Ignore PAD targets
    # --------------------------------

    loss = criterion(
        logits.reshape(
            -1,
            logits.size(-1)
        ),
        target_tokens.reshape(-1)
    )

    return loss, logits

Test: